# 05 — Met Office Weather DataHub (Land Observations)

Fetches Met Office **Land Observations** for each configured site:
1) `/nearest` by lat/lon (gets nearest observing station + geohash)
2) `/{geohash}` for observations

Outputs:
- `outputs/05_metoffice_datahub_weather/raw/*.json`
- `outputs/05_metoffice_datahub_weather/derived/nearest_index.json`
- `outputs/05_metoffice_datahub_weather/manifest.json`


In [ ]:
import os, json, hashlib
from pathlib import Path
from datetime import datetime, timezone
import requests
import yaml

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def sha256_file(p: Path) -> str:
    return sha256_bytes(p.read_bytes())

def write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")

def env(name: str) -> str | None:
    v = os.getenv(name)
    return v.strip() if isinstance(v, str) and v.strip() else None

def manifest_base(step: str, config_paths=None):
    m = {
        "step": step,
        "created_at_utc": utc_now(),
        "configs": [],
        "requests": [],
        "artifacts": []
    }
    for cp in (config_paths or []):
        if cp and cp.exists():
            m["configs"].append({"path": str(cp), "sha256": sha256_file(cp)})
    return m

def add_artifact(manifest: dict, path: Path, kind: str, meta=None):
    if path.exists():
        item = {"kind": kind, "path": str(path), "sha256": sha256_file(path)}
        if meta:
            item["meta"] = meta
        manifest["artifacts"].append(item)

def record_request(manifest: dict, *, url: str, params: dict | None, status: int, ok: bool, out_path: Path | None):
    rec = {
        "ts_utc": utc_now(),
        "url": url,
        "params": params or {},
        "status": status,
        "ok": bool(ok),
    }
    if out_path and out_path.exists():
        rec["response_sha256"] = sha256_file(out_path)
        rec["response_path"] = str(out_path)
    manifest["requests"].append(rec)

# --- resolve repo root robustly ---
cwd = Path.cwd().resolve()

def find_repo_root(start: Path) -> Path:
    cur = start
    for _ in range(10):
        if (cur / "configs").exists() and (cur / "notebooks").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start

ROOT = find_repo_root(cwd)
CONFIGS = ROOT / "configs"
OUTPUTS = ROOT / "outputs"

print("UTC now:", utc_now())
print("CWD:", cwd)
print("ROOT:", ROOT)
print("CONFIGS exists:", CONFIGS.exists(), "OUTPUTS exists:", OUTPUTS.exists())


In [ ]:
step = "05_metoffice_datahub_weather"
OUTDIR = OUTPUTS / step
RAW = OUTDIR / "raw"
DERIVED = OUTDIR / "derived"
RAW.mkdir(parents=True, exist_ok=True)
DERIVED.mkdir(parents=True, exist_ok=True)

cfg_path = CONFIGS / "run.yml"
if not cfg_path.exists():
    raise FileNotFoundError(f"configs/run.yml not found at: {cfg_path}")

cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8")) or {}

# --- normalize sites (supports list or dict) ---
sites_norm = []

if isinstance(cfg.get("sites"), list):
    for s in cfg.get("sites", []):
        sid = s.get("id") or s.get("site_id") or s.get("name")
        sites_norm.append({
            "site_id": sid,
            "name": s.get("name"),
            "lat": s.get("lat"),
            "lon": s.get("lon"),
            "role": (s.get("role") or "context")
        })
elif isinstance(cfg.get("sites"), dict):
    for k, v in (cfg.get("sites") or {}).items():
        sites_norm.append({
            "site_id": k,
            "name": v.get("name"),
            "lat": v.get("lat"),
            "lon": v.get("lon"),
            "role": (v.get("role") or "context")
        })

if isinstance(cfg.get("controls"), dict):
    for k, v in (cfg.get("controls") or {}).items():
        sites_norm.append({
            "site_id": k,
            "name": v.get("name"),
            "lat": v.get("lat"),
            "lon": v.get("lon"),
            "role": "control"
        })

usable = [s for s in sites_norm if s.get("site_id") and s.get("lat") is not None and s.get("lon") is not None]
print("Sites usable:", len(usable))
for s in usable:
    print("-", s["site_id"], s.get("name"), s.get("lat"), s.get("lon"), s.get("role"))


In [ ]:
# --- Met Office API configuration ---
# Prefer a dedicated secret name if you create one; otherwise fall back to MET_OFFICE_KEY / METOFFICE_API_KEY.
APIKEY = env("MET_OFFICE_LAND_OBSERVATIONS") or env("MET_OFFICE_KEY") or env("METOFFICE_API_KEY")
if not APIKEY:
    raise RuntimeError("Missing Met Office API key. Set MET_OFFICE_LAND_OBSERVATIONS (preferred) or MET_OFFICE_KEY / METOFFICE_API_KEY.")

# Land Observations base URL (from your screenshots)
BASE_URL = "https://data.hub.api.metoffice.gov.uk/observation-land/1"

SESSION = requests.Session()
SESSION.headers.update({"apikey": APIKEY})

def save_response(path: Path, r: requests.Response):
    ct = (r.headers.get("content-type") or "").lower()
    if "application/json" in ct:
        try:
            write_json(path, r.json())
            return
        except Exception:
            pass
    # fallback
    write_json(path, {
        "status": "nonjson_or_parse_error",
        "http_status": r.status_code,
        "content_type": r.headers.get("content-type"),
        "text": r.text[:20000]
    })

def get_json(url: str, params: dict | None, out_path: Path, manifest: dict):
    r = SESSION.get(url, params=params, timeout=30)
    save_response(out_path, r)
    record_request(manifest, url=url, params=params, status=r.status_code, ok=r.ok, out_path=out_path)
    return r


In [ ]:
manifest = manifest_base(step, config_paths=[cfg_path])

nearest_index = []

for s in usable:
    site_id = s["site_id"]
    lat = s["lat"]
    lon = s["lon"]

    # 1) nearest
    nearest_url = f"{BASE_URL}/nearest"
    nearest_path = RAW / f"metoffice_nearest_{site_id}.json"
    r1 = get_json(nearest_url, {"lat": lat, "lon": lon}, nearest_path, manifest)

    geohash = None
    try:
        j1 = json.loads(nearest_path.read_text(encoding="utf-8"))
        # The sample files imply a "geohash" field; keep it defensive.
        geohash = j1.get("geohash") or j1.get("nearest") or j1.get("data", {}).get("geohash")
        if isinstance(geohash, dict):
            geohash = geohash.get("geohash")
    except Exception:
        geohash = None

    nearest_index.append({
        "site_id": site_id,
        "lat": lat,
        "lon": lon,
        "nearest_http_ok": bool(r1.ok),
        "nearest_http_status": r1.status_code,
        "geohash": geohash,
        "nearest_raw": str(nearest_path)
    })

    # 2) observations for geohash (best effort)
    if geohash:
        obs_url = f"{BASE_URL}/{geohash}"
        obs_path = RAW / f"metoffice_obs_{site_id}.json"
        r2 = get_json(obs_url, None, obs_path, manifest)
    else:
        # still record artifact for diagnostics
        diag = {
            "site_id": site_id,
            "note": "No geohash extracted from /nearest response; observations not fetched.",
        }
        obs_path = RAW / f"metoffice_obs_{site_id}.json"
        write_json(obs_path, diag)

write_json(DERIVED / "nearest_index.json", nearest_index)

# provenance artifacts
add_artifact(manifest, DERIVED / "nearest_index.json", "nearest_index")
for item in nearest_index:
    add_artifact(manifest, Path(item["nearest_raw"]), "raw_nearest")
    obs_p = RAW / f"metoffice_obs_{item['site_id']}.json"
    add_artifact(manifest, obs_p, "raw_observations", meta={"geohash": item.get("geohash")})

write_json(OUTDIR / "manifest.json", manifest)

print("Wrote:", DERIVED / "nearest_index.json")
print("Wrote:", OUTDIR / "manifest.json")
print("Done.")
